In [1]:
pip install librosa

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import h5py
import librosa
import numpy as np
import pandas as pd
import torch
# PC02a
# import models as md
# PC02b
import model_optuna as md

TEST_FOLDER = '/mnt/share/Data-WearableAcoustic/Pre/Test'
OUT_FOLDER = 'Data-Predictions'
os.makedirs(OUT_FOLDER, exist_ok=True)

device = torch.device('cpu')
#device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


In [ ]:
# PC02A
model = md.CNN_Improved((1, 128, 126), 4).to(device)
model.load_state_dict(torch.load('_best_cnn_model.pth', map_location=device))
model.eval()

# PC02b
#model = md.CNN_Optuna((1, 128, 126), 4, n_blocks=2, no_features=8).to(device)
#model.load_state_dict(torch.load('_best_cnn_optuna.pth', map_location=device))
#model.eval()
print("Model loaded successfully")

Model loaded successfully


In [ ]:
window_sec = 1.0
step_sec = 0.5

# Added batch_size to prevent NVML/CUDA Out-of-Memory errors
batch_size = 64 

test_files = sorted([f for f in os.listdir(TEST_FOLDER) if f.endswith('.h5')])
print(test_files)

for file_name in test_files:
    print(f'Processing {file_name} ...')

    file_path = os.path.join(TEST_FOLDER, file_name)
    out_file = os.path.join(OUT_FOLDER, file_name.replace('.h5', '.csv'))

    with h5py.File(file_path, 'r') as f:
        audio = f['audio_data'][:].astype(np.float32)
        audio_sr = int(np.array(f['audio_sample_rate']))
        label_time = f['label_time'][:]
        label_sr = int(np.array(f['label_sample_rate']))

    audio_win_len = int(window_sec * audio_sr)
    audio_hop_len = int(step_sec * audio_sr)

    windows = []
    spans = []

    n_windows = (len(audio) - audio_win_len) // audio_hop_len + 1

    for i in range(n_windows):
        audio_start = i * audio_hop_len
        audio_end = audio_start + audio_win_len
        segment_audio = audio[audio_start:audio_end]

        mel = librosa.feature.melspectrogram(
            y=segment_audio.astype(float),
            sr=audio_sr,
            n_fft=1024,
            hop_length=128,
            n_mels=128
        )
        mel = librosa.power_to_db(mel, ref=np.max)

        windows.append(mel)

        label_start = int(audio_start * label_sr / audio_sr)
        label_end = int(audio_end * label_sr / audio_sr)
        spans.append((label_start, min(label_end, len(label_time))))

    windows = np.array(windows, dtype=np.float32)[:, None, :, :]

    # Initializing empty list for predictions to allow for batching
    all_preds = [] 

    # Inference moved inside a batching loop to ensure CUDA/CPU compliance
    model.eval()
    with torch.no_grad():
        for i in range(0, len(windows), batch_size):
            # [corr] Create tensor and move only the small batch to device (CPU or GPU)
            X_batch = torch.tensor(windows[i : i + batch_size], dtype=torch.float32).to(device) 
            
            # [corr] Forward pass on the small batch
            outputs = model(X_batch) 
            
            # [corr] Move results back to CPU for numpy compatibility and append
            batch_preds = torch.argmax(outputs, dim=1).cpu().numpy() 
            all_preds.extend(batch_preds)
            
            # [corr] Optional memory cleanup
            del X_batch 

    #  Convert the full list of predictions back to a numpy array for voting
    preds = np.array(all_preds) 

    votes = np.zeros((len(label_time), 4), dtype=np.uint16)

    for pred, (start, end) in zip(preds, spans):
        votes[start:end, pred] += 1

    full_preds = np.argmax(votes, axis=1).astype(int)

    pd.DataFrame(full_preds).to_csv(out_file, index=False, header=False)

    print(f'Saved {out_file} | labels: {np.unique(full_preds)} | length: {len(full_preds)}')

['s11_trial1.h5', 's11_trial2.h5', 's11_trial3.h5', 's12_trial1.h5', 's12_trial2.h5', 's12_trial3.h5', 's13_trial1.h5', 's13_trial2.h5', 's13_trial3.h5']
Processing s11_trial1.h5 ...
Saved Data-Predictions/s11_trial1.csv | labels: [0 1 2 3] | length: 7146625
Processing s11_trial2.h5 ...
Saved Data-Predictions/s11_trial2.csv | labels: [0 1 2 3] | length: 6924950
Processing s11_trial3.h5 ...
Saved Data-Predictions/s11_trial3.csv | labels: [0 1 2 3] | length: 7045450
Processing s12_trial1.h5 ...
Saved Data-Predictions/s12_trial1.csv | labels: [0 1 2 3] | length: 7676148
Processing s12_trial2.h5 ...
Saved Data-Predictions/s12_trial2.csv | labels: [0 1 2 3] | length: 7512373
Processing s12_trial3.h5 ...
Saved Data-Predictions/s12_trial3.csv | labels: [0 1 2 3] | length: 7438071
Processing s13_trial1.h5 ...
Saved Data-Predictions/s13_trial1.csv | labels: [0 1 2 3] | length: 7531470
Processing s13_trial2.h5 ...
Saved Data-Predictions/s13_trial2.csv | labels: [0 1 2 3] | length: 8516621
Proces